# 0908 7일차

## 0. 파이썬 문법: 속성 접근(`.`)과 키 접근(`[]`)

`keras21`에서 같은 데이터를 두 방법으로 꺼냄

```python
x = datasets.data       # 속성(attribute) 접근
x = datasets['data']    # 키(key) 접근
```

→ **둘 다 되고 결과도 같음.** 원래는 서로 다른 문법인데도

| 표기 | 대상 | 예시 |
|---|---|---|
| `.이름` | **객체의 속성** | `df.shape`, `model.fit()`, `hist.history` |
| `['이름']` | **딕셔너리의 키** | `hist.history['loss']`, `submission['count']` |

### `Bunch`는 둘 다 되는 특수한 타입

sklearn이 데이터셋을 담아 돌려주는 그릇이 **`Bunch`** 인데, **딕셔너리를 상속한 클래스**임

```python
print(isinstance(datasets, dict))   # True  <- 딕셔너리이기도 함
```

→ 딕셔너리라서 `datasets['data']`가 되고, 키를 속성처럼도 꺼내도록 만들어놔서 `datasets.data`도 됨

→ **sklearn 데이터셋에서만 통하는 편의 기능.** 일반 딕셔너리에 `.`을 쓰면 에러

```python
d = {'loss': [0.5, 0.3]}
d['loss']    # OK
d.loss       # AttributeError: 'dict' object has no attribute 'loss'
```

### cf) `type()`으로 정체 확인

```python
print(type(x))          # <class 'numpy.ndarray'>
print(type(datasets))   # <class 'sklearn.utils._bunch.Bunch'>
```

→ `type()`은 **파이썬 내장 함수**. "이게 뭐지" 싶을 때 가장 먼저 찍어볼 것

→ `datasets.data`는 판다스가 아니라 **넘파이 배열** (3일차 §4)

## 1. Early Stopping - 최고 모델을 보장하지 않음

### val_loss 최저 ≠ test loss 최저

`restore_best_weights=True`는 **`val_loss`가 가장 낮았던 가중치**로 되돌려줌

→ 그런데 최종 `evaluate` 결과(= **test** loss)는 오히려 나빠질 수 있음

| 원인 | 설명 |
|---|---|
| **val ≠ test** | 되돌린 기준은 `val_loss` 최저점. `x_test`는 또 다른 데이터 |
| **`patience`가 짧음** | 더 내려갈 수 있는데 일찍 끊김 → 과소적합 |
| **`restore_best_weights`가 꺼짐** | 기본값 `False`면 최저점에서 후퇴한 상태로 끝남 |
| **실행마다 다름** | 분할, 가중치 초기값 등 |

→ Early Stopping은 **불필요하게 오래 학습하지 않게** 하는 장치

### local minima와 global minima

경사 하강법은 기울기가 0인 곳에서 멈추는데, **그런 지점이 여러 개**일 수 있음

```
loss
  │    \                    /
  │     \      ___         /
  │      \    /   \       /
  │       \__/     \     /
  │      local      \___/
  │      minima      global minima
  └──────────────────────────────→ w
```

| | 뜻 |
|---|---|
| **local minima** | 주변보다는 낮지만 **전체 최저점은 아닌** 지점 |
| **global minima** | 전체에서 가장 낮은 지점 |

→ 학습률이 작으면 처음 만난 local minima에서 멈출 수 있음. `adam`이 관성을 갖는 이유 중 하나 (3일차 cf)

### loss 0 - 경고 신호

| loss가 0에 가까울 때 | 의심할 것 |
|---|---|
| 훈련 데이터에서만 0 | **과적합** - 훈련 데이터를 외움 |
| test에서도 0 | **데이터 누수** — 정답이 특성에 섞임 (5일차 §2) |
| 애초에 도달 불가 | **노이즈**가 있으면 어떤 모델도 0이 될 수 없음 (3일차 §3) |

→ 목표는 loss 0이 아니라 **`val_loss`가 최저인 지점**. Early Stopping이 `val_loss`를 감시하는 이유

## 2. 회귀에서 분류로

4일차 §1에서 갈래만 정리했던 **분류(classification)**를 오늘부터 실제로 다룸

| | 회귀 | 분류 |
|---|---|---|
| 예측하는 것 | **연속적인 수치** | 정해진 **라벨** 중 하나 |
| 예시 | 집값, 대여량 | 악성/양성, 개/고양이 |
| 답 | 197.3처럼 **딱 떨어지지 않음** | **딱 떨어짐** |

→ 분류 모델은 **학습한 라벨 밖의 답을 낼 수 없음.** 정해진 선택지 밖은 처리할 방법이 없어서, **라벨 목록을 먼저 확정**하고 시작함

### 이진분류와 다중분류

| | 라벨 개수 | 출력층 | loss |
|---|---|---|---|
| **이진분류** | 2개 | `Dense(1, activation='sigmoid')` | `binary_crossentropy` |
| **다중분류** | 3개 이상 | `Dense(라벨수, activation='softmax')` | `categorical_crossentropy` |

→ 오늘 다루는 유방암 데이터는 라벨이 `0`(악성) / `1`(양성) 두 개라 **이진분류**

## 3. 분류 데이터를 받으면 먼저 할 일

`keras21_sigmoid_metrics_cancer.py` — sklearn 유방암 데이터 (569개, 특성 30개)

### 1) 라벨이 무엇인지 확인

```python
print(np.unique(y))                 # [0 1]
print(datasets.target_names)        # ['malignant' 'benign']  악성 / 양성
```

→ 라벨이 몇 개이고 무슨 값인지에 따라 **출력층과 loss가 정해지므로** 항상 먼저 확인

### 2) 라벨별 개수 세기

```python
print(np.unique(y, return_counts=True))   # (array([0, 1]), array([212, 357]))
print(pd.Series(y).value_counts())
print(pd.DataFrame(y).value_counts())
```

| 방법 | 라이브러리 | 비고 |
|---|---|---|
| `np.unique(y, return_counts=True)` | 넘파이 | 값 배열과 개수 배열을 **튜플**로 |
| `pd.Series(y).value_counts()` | 판다스 | 개수 많은 순으로 정렬 |

→ 판다스의 자료구조는 **`DataFrame`(표)과 `Series`(한 줄)** 두 가지뿐

### 3) 불균형한지 판단

```
0 (악성) : 212개  = 37.3%
1 (양성) : 357개  = 62.7%
```

→ 한쪽으로 심하게 쏠리면 모델이 **많은 쪽만 찍어도 정확도가 높게** 나옴

→ 99:1 데이터면 전부 다수 클래스로 찍어도 정확도 99%

→ 지금은 37:63이라 심각하진 않지만 **확인하는 습관**이 중요

### 4) `stratify`로 비율을 지키며 분할

```python
x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.7, random_state=234,
    stratify=y,        # y의 라벨 비율을 train/test 양쪽에 유지
)
```

`random_state`를 0~199까지 바꿔가며 train의 악성 비율을 측정한 결과

| | train의 0(악성) 비율 | 폭 |
|---|---|---|
| `stratify` 없음 | 33.9% ~ 41.0% | **7.0%p** |
| `stratify=y` | 37.2% ~ 37.2% | **0.0%p (고정)** |

→ 운 나쁜 분할에서는 훈련 데이터의 라벨 구성이 원본과 꽤 달라짐

→ **평가 데이터는 좀 불균형해도 되지만, 훈련 데이터가 쏠리면 학습 자체가 편향됨**

#### cf) `stratify` - 회귀에서는 못 씀

`stratify`는 **라벨이 이산적일 때만** 동작. 집값처럼 연속값을 넣으면 에러가 남

→ 그래서 3~6일차 회귀 실습에는 이 인자가 없었음

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

x, y = load_breast_cancer(return_X_y=True)

print(np.unique(y, return_counts=True))     # (array([0, 1]), array([212, 357]))
print(pd.Series(y).value_counts())          # 1 -> 357,  0 -> 212

# stratify 유무에 따른 train 라벨 비율 비교
for st in (None, y):
    ratios = []
    for seed in range(200):
        _, _, y_tr, _ = train_test_split(x, y, train_size=0.7,
                                         random_state=seed, stratify=st)
        ratios.append((y_tr == 0).mean() * 100)
    tag = "stratify=y   " if st is not None else "stratify 없음"
    print(f"{tag}  최소 {min(ratios):.1f}%  최대 {max(ratios):.1f}%")

# stratify 없음  최소 33.9%  최대 41.0%   <- 분할 운에 따라 흔들림
# stratify=y     최소 37.2%  최대 37.2%   <- 원본 비율(37.3%)에 고정

## 4. 출력층은 `sigmoid`

```python
model.add(Dense(32, input_dim=30, activation='relu'))
...
model.add(Dense(1, activation='sigmoid'))   # 이진분류의 출력층
```

→ 은닉층은 회귀와 같이 `relu` (5일차 §3), **출력층만 `sigmoid`**

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

```
      y ↑
    1 ─┤            ╭────────
       │         ╭──╯
  0.5 ─┤      ╭──╯
       │   ╭──╯
    0 ─┼───╯──────────────────→ x
                  0
```

| 성질 | 설명 |
|---|---|
| 출력 범위 | 항상 **0과 1 사이** |
| `x = 0`일 때 | 정확히 `0.5` |
| 해석 | "1일 **확률**"로 읽을 수 있음 |

### sigmoid - 필요한 이유 (실측)

같은 구조·시드로 **출력층과 loss만** 바꿔 유방암 test 171개를 예측

| 구성 | 예측값 범위 | 정확도 |
|---|---|---|
| `sigmoid` + `binary_crossentropy` | **0.000 ~ 0.999** | **0.918** |
| `linear` + `mse` (회귀식 그대로) | **-6.091 ~ 7.071** | 0.485 |

→ 정답은 `0` 아니면 `1`인데 예측이 `-6.09`, `7.07`이면 **확률로 읽을 수 없음**

### cf) sigmoid가 0과 1을 만들어주는 것은 아님

**0.5로 자르는 건 accuracy가 하는 일**

```
입력 z          : [-4.0   -1.0    0.0    0.4    1.0    4.0 ]
sigmoid(z)      : [ 0.018  0.269  0.500  0.599  0.731  0.982]   <- 연속값 그대로
0.5 기준 반올림 : [ 0      0      1      1      1      1    ]   <- 여기서 0/1이 됨
```

→ `model.predict()`도 `0.9997`처럼 **소수**를 돌려줌

→ 케라스 `BinaryAccuracy`가 **`threshold=0.5`**를 기본으로 갖고 있어 정확도 계산 시 잘라줌

```python
y_class = (y_predict > 0.5).astype(int)      # 0/1이 필요하면 직접 변환
```

## 5. 컴파일 - `binary_crossentropy`와 `metrics`

```python
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['acc'],     # 'accuracy'와 같음 (별칭)
             )
```

| 문제 | loss |
|---|---|
| 회귀 | `mse`, `mae` |
| **이진분류** | **`binary_crossentropy`** |
| 다중분류 | `categorical_crossentropy` |

→ "정답이 1인데 0.1이라고 답했다" 같은 **확률의 어긋남**에 벌점을 매김

### loss와 metrics의 차이

| | loss | metrics |
|---|---|---|
| 쓰임 | **가중치 갱신에 사용** | **사람이 보기만** 함 |
| 미분 | **가능해야 함** | 필요 없음 |
| 개수 | 하나 | 여러 개 가능 |

→ 정확도를 loss로 못 쓰는 이유: **정확도는 계단 함수라 미분이 안 됨**

→ 0.51이 0.52가 되어도 정확도는 그대로 → 기울기가 0 → 갱신 방향을 알 수 없음

→ **훈련은 `binary_crossentropy`, 해석은 정확도로** (4일차 §2의 "훈련은 mse, 해석은 RMSE"와 같은 구도)

### `metrics`를 주면 evaluate가 리스트를 돌려줌

```python
loss = model.evaluate(x_test, y_test)
print(loss)   # [0.21258428692817688, 0.9356725215911865]
              #   └ binary_crossentropy   └ accuracy
```

→ 변수 이름이 `loss`인데 실제로는 리스트라 헷갈리기 쉬움

```python
loss, acc = model.evaluate(x_test, y_test)   # 이렇게 받으면 명확
```

### cf) 분류에 R²와 RMSE를 쓰면

`keras21` 아래쪽에서 `r2_score`와 `RMSE`를 계산하는데, **분류에서는 해석이 어려운 값**

```
r2 :   0.7987      <- 회귀용 지표라 "설명력"으로 읽기 애매
rmse : 0.2171      <- 0/1 정답과 확률 예측의 차이일 뿐
```

→ 계산은 되지만 **분류의 성적표는 정확도**로 읽어야 함

→ 나중에 배울 정밀도·재현율·혼동행렬이 분류용 지표

→ 특히 **암 진단**처럼 "악성을 양성으로 잘못 보는 것"이 훨씬 위험하면 정확도만으로도 부족

In [ ]:
import numpy as np

# sigmoid는 0/1이 아니라 0~1 사이 연속값을 돌려준다
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.array([-4.0, -1.0, 0.0, 0.4, 1.0, 4.0])

print("sigmoid(z)      :", np.round(sigmoid(z), 3))
# [0.018 0.269 0.5   0.599 0.731 0.982]

print("0.5 기준 반올림 :", (sigmoid(z) >= 0.5).astype(int))
# [0 0 1 1 1 1]   <- 이 단계는 accuracy(threshold=0.5)가 하는 일

# predict 결과를 직접 0/1로 바꾸려면
y_predict = np.array([[0.9997], [0.0012], [0.6310]])
print("변환 후:", (y_predict > 0.5).astype(int).ravel())   # [1 0 1]